# Netflix 

Every sample below is a small, generic function. None of them are tied to
one specific column name more than necessary, so you can reuse the same
pattern throughout the project. Each one is followed by a short explanation of
*why* that factor matters when you're trying to predict whether a new TV
series concept has hit potential.

The composite dataset has one row per matched Netflix title, with these
columns:

| column | meaning |
|---|---|
| `netflix_title` | the title as it appears in Netflix's Top 10 data |
| `netflix_viewing_hours` | total hours viewed while the title was on the Top 10 list |
| `netflix_weeks` | number of weeks the title spent on the Top 10 list |
| `netflix_year_hint` | the year the title first appeared on the list |
| `tmdb_title`, `tmdb_popularity`, `tmdb_vote_average`, `tmdb_vote_count` | matched TMDB metadata |
| `imdb_title`, `imdb_averageRating`, `imdb_numVotes` | matched IMDb metadata |

A row can have missing `tmdb_*` or `imdb_*` values — the fuzzy-matching
pipeline from Module 2 doesn't find a confident match for every title, so
real-world coverage is never 100%.



In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.unicode_minus"] = False

DATA_DIR = Path("./data")

netflix = pd.read_csv(DATA_DIR / "netflix.titles.composite.csv")
imdb = pd.read_csv(DATA_DIR / "imdb.titles.composite.csv", low_memory=False)
tmdb = pd.read_csv(DATA_DIR / "tmdb.titles.v3.csv", low_memory=False)

print("netflix:", netflix.shape)
print("imdb   :", imdb.shape)
print("tmdb   :", tmdb.shape)
netflix.head(3)

netflix: (691, 13)
imdb   : (299045, 15)
tmdb   : (168639, 29)


,key,netflix_viewing_hours,netflix_weeks,netflix_year_hint,netflix_title,netflix_clean_title,tmdb_title,tmdb_popularity,tmdb_vote_average,tmdb_vote_count,imdb_title,imdb_averageRating,imdb_numVotes
0,10,26580000,2,2021,The 100,the 100,The 100,127.224,7.916,7666.0,The 100,9.4,12.0
1,10_fr_mi,13590000,2,2021,1000 Miles from Christmas,1000 miles from christmas,100 Miles From Nowhere,1.400,3.600,5.0,1000 Miles from Christmas,5.9,2334.0
2,10_lo,12600000,1,2021,Love 101,love 101,Love 101,14.518,7.974,154.0,Love 101,5.0,159.0


# Part 1 — Data Preprocessing & Integration

Before modeling, inspect the data quality and determine what one row in your final analytical dataset should represent.

Pay particular attention to:
- missing IMDb/TMDB matches;
- one-to-many matches after merging;
- duplicated titles or records;
- whether duplicated rows could bias later analysis;
- whether the same title could leak into both training and testing sets.


In [6]:
missing_tmdb = netflix["tmdb_title"].isna().sum()
missing_imdb = netflix["imdb_title"].isna().sum()

both_missing = netflix[netflix["tmdb_title"].isna() & netflix["imdb_title"].isna()].shape[0]
both_present = netflix[netflix["tmdb_title"].notna() & netflix["imdb_title"].notna()].shape[0]

print(f"Missing tmdb_title: {missing_tmdb} / {len(netflix)}")
print(f"Missing imdb_title: {missing_imdb} / {len(netflix)}")
print(f"Missing in both sources: {both_missing}")
print(f"Present in both sources: {both_present}")

df_clean = netflix[netflix["tmdb_title"].notna() & netflix["imdb_title"].notna()].copy()
print(f"Before cleaning: {len(netflix)}")
print(f"After cleaning : {len(df_clean)} ({len(df_clean) / len(netflix):.1%} retained)")

print("\nAll column names:")
print(df_clean.columns.tolist())

Missing tmdb_title: 297 / 691
Missing imdb_title: 26 / 691
Missing in both sources: 23
Present in both sources: 391
Before cleaning: 691
After cleaning : 391 (56.6% retained)

All column names:
['key', 'netflix_viewing_hours', 'netflix_weeks', 'netflix_year_hint', 'netflix_title', 'netflix_clean_title', 'tmdb_title', 'tmdb_popularity', 'tmdb_vote_average', 'tmdb_vote_count', 'imdb_title', 'imdb_averageRating', 'imdb_numVotes']


**Interpretation.** The core file has substantially more missing TMDb matches than IMDb matches. Keeping only titles matched to both sources leaves 391 records. The removed titles are not necessarily random: movies cannot appear in the TV-only IMDb/TMDb source files, and obscure or differently titled works may also fail matching. This can introduce selection bias.

In [7]:
original_rows = len(df_clean)

imdb_extra = imdb[
    [
        "primaryTitle",
        "tconst",
        "titleType",
        "originalTitle",
        "startYear",
        "endYear",
        "runtimeMinutes",
        "genres",
        "all_akas",
        "cast",
        "directors",
        "writers"
    ]
].copy()

df_clean = df_clean.merge(
    imdb_extra,
    left_on="imdb_title",
    right_on="primaryTitle",
    how="left"
)

tmdb_extra = tmdb[
    [
        "name",
        "id",
        "number_of_seasons",
        "number_of_episodes",
        "original_language",
        "overview",
        "adult",
        "backdrop_path",
        "first_air_date",
        "last_air_date",
        "homepage",
        "in_production",
        "original_name",
        "poster_path",
        "type",
        "status",
        "tagline",
        "genres",
        "created_by",
        "languages",
        "networks",
        "origin_country",
        "spoken_languages",
        "production_companies",
        "production_countries",
        "episode_run_time"
    ]
].copy()


df_clean = df_clean.merge(
    tmdb_extra,
    left_on="tmdb_title",
    right_on="name",
    how="left"
)

print(f"Rows before merging: {original_rows}")
print(f"Rows after merging : {len(df_clean)}")

if len(df_clean) == original_rows:
    print("Row count unchanged.")
else:
    print("WARNING: Row count changed!")

print( df_clean.shape)
df_clean.head(3)

Rows before merging: 391
Rows after merging : 801
(801, 51)


,key,netflix_viewing_hours,netflix_weeks,netflix_year_hint,netflix_title,netflix_clean_title,tmdb_title,tmdb_popularity,tmdb_vote_average,tmdb_vote_count,imdb_title,imdb_averageRating,imdb_numVotes,primaryTitle,tconst,...,in_production,original_name,poster_path,type,status,tagline,genres_y,created_by,languages,networks,origin_country,spoken_languages,production_companies,production_countries,episode_run_time
0,10,26580000,2,2021,The 100,the 100,The 100,127.224,7.916,7666.0,The 100,9.4,12.0,The 100,tt2661044,...,False,The 100,/wcaDIAG1QdXQLRaj4vC1EFdBT2.jpg,Scripted,Ended,Survival isn't who you are. It's who you become.,"Sci-Fi & Fantasy, Drama, Action & Adventure",Jason Rothenberg,en,The CW,US,English,"Warner Bros. Television, Alloy Entertainment, ...",United States of America,43
1,10_fr_mi,13590000,2,2021,1000 Miles from Christmas,1000 miles from christmas,100 Miles From Nowhere,1.400,3.600,5.0,1000 Miles from Christmas,5.9,2334.0,NaN,NaN,...,True,100 Miles From Nowhere,/pK2p1DvPmW50pgc9N10lDwwwGQi.jpg,Documentary,Returning Series,NaN,Documentary,NaN,en,Animal Planet,US,English,Renegade83,NaN,42
2,10_lo,12600000,1,2021,Love 101,love 101,Love 101,14.518,7.974,154.0,Love 101,5.0,159.0,Love 101,tt10516352,...,False,Aşk 101,/4OOhHLZeTtNji4V6ZRYHQSvjlZc.jpg,Scripted,Ended,NaN,"Comedy, Drama",Meriç Acemi,tr,Netflix,TR,Türkçe,Ay Yapım,Turkey,45


(1) Explain why the number of rows increased after the merge.

(2) Compare the dataset before and after merging. Please identify and explain any important changes you observe, such as:

•	changes in the number of rows or columns;

•	duplicated records;

•	missing IMDb or TMDB information.


(3) After merging the IMDb and TMDB datasets with the cleaned Netflix dataset, you observe that the resulting dataset may differ from the original dataset in terms of the number of rows, columns, and missing values.

(3-1) Explain the problem you identified.

(3-2) Explain how you solve this problem.

There is no single required implementation. Your solution should be justified based on your analysis of the data and the requirements of the subsequent machine learning task.


In [ ]:
# (3) Implement your code here

## Part 2 — Feature Engineering Examples 1 : Binge velocity: hours viewed per week on the list

Two shows can rack up the same total viewing hours in very different ways:
one gets devoured in two weeks, another lingers modestly for ten. Dividing
total hours by weeks-on-list surfaces that difference as a single number —
a rough proxy for how quickly word-of-mouth spreads. For a writer, this is
a hint that pacing and "bingeability" (short arcs, strong episode-ending
hooks) may matter as much as overall quality.


In [8]:
def binge_velocity(frame: pd.DataFrame,
                    hours_col: str = "netflix_viewing_hours",
                    weeks_col: str = "netflix_weeks") -> pd.Series:
    """Average viewing hours per week on the Top 10 list.

    Guards against division by zero for any title with weeks_col == 0.
    """
    weeks = frame[weeks_col].replace(0, np.nan)
    return (frame[hours_col] / weeks).rename("binge_velocity")


df_clean["binge_velocity"] = binge_velocity(df_clean)
df_clean[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "binge_velocity"]].head(3)

,netflix_title,netflix_viewing_hours,netflix_weeks,binge_velocity
0,The 100,26580000,2,13290000.0
1,1000 Miles from Christmas,13590000,2,6795000.0
2,Love 101,12600000,1,12600000.0


## Part 2 —Feature Engineering Examples 2 : A trustworthy rating: shrink small samples toward the mean

A 9.4 rating from 12 IMDb votes and a 7.8 rating from 200,000 votes are not
equally informative, but a plain average treats them the same. This is the
same problem IMDb's own historical "weighted rating" formula addressed:
blend each title's rating with a prior (e.g. the dataset-wide mean),
weighted by how many votes back it up. A title with few votes gets pulled
toward the average; a title with many votes keeps its own score. This gives
a much more defensible "quality" feature than the raw rating column.


In [9]:
def bayesian_rating(frame: pd.DataFrame,
                     rating_col: str,
                     votes_col: str,
                     prior_votes: int = 500) -> pd.Series:
    """Shrink each rating toward the dataset mean based on vote count.

    weighted_rating = (v / (v + m)) * R + (m / (v + m)) * C

    where R is the title's own rating, v its vote count, C the dataset-wide
    mean rating, and m the `prior_votes` "confidence" threshold: titles with
    v << m get pulled hard toward C, titles with v >> m barely move.
    """
    v = frame[votes_col]
    R = frame[rating_col]
    C = frame[rating_col].mean()
    m = prior_votes
    return ((v / (v + m)) * R + (m / (v + m)) * C).rename(f"{rating_col}_shrunk")


df_clean["imdb_rating_shrunk"] = bayesian_rating(df_clean, "imdb_averageRating", "imdb_numVotes")
df_clean[["netflix_title", "imdb_averageRating", "imdb_numVotes", "imdb_rating_shrunk"]].dropna().head(3)


,netflix_title,imdb_averageRating,imdb_numVotes,imdb_rating_shrunk
0,The 100,9.4,12.0,6.684929
1,1000 Miles from Christmas,5.9,2334.0,6.026988
2,Love 101,5.0,159.0,6.228959


## Part 3 - Define HIT

**This is the main open-ended analysis section.**

You have to:

**Step 1:** Explore variables that may be related to title success.

**Step 2:** Create meaningful features based on your analysis.

**Step 3:** Define your own numerical **HitScore** using selected variables and/or engineered features.

**Step 4:** Explain your feature selection and any transformations, scaling, or weighting strategies you use.

**Step 5:** Choose and justify a reasonable threshold to define **Hit (1)** and **Not Hit (0)**.

There is **no single correct HitScore formula**. Your definition should be reasonable, interpretable, and supported by your analysis.


After completing your data analysis, feature engineering, and HitScore definition, answer the following questions:

(1) Explain your data analysis and feature engineering.
What variables did you analyze or create, and why did you choose them? Briefly explain how your analysis or engineered features help describe the success of a Netflix title.

(2) What insights did you discover from your analysis?
Describe the key patterns, relationships, or findings you observed from the data. Support your insights with appropriate statistics, tables, or visualizations when possible.

Based on your analysis, define your own HitScore and use it to classify each title as Hit (1) or Not Hit (0). Clearly explain the reasoning behind your definition.

## Part 4 - Modeling
In this part, you will train and evaluate a classification model to predict Hit (1) or Not Hit (0).

Step 1 — Split the Dataset

Split your final dataset into:Training set: 80% / Test set: 20%

After splitting, display the class distribution of both datasets


Step 2 — Perform Cross-Validation

Perform 3-fold cross-validation on the training set only.

Report the cross-validation results for: Accuracy, Precision, Recall, F1-score

Report the mean and standard deviation across the three folds.

Do not use the test set during cross-validation.

Step 3 —  Evaluate on the Test Set

Use the trained model to make predictions on the held-out test set (20%).

Report: Accuracy, Precision, Recall, F1-score, Confusion Matrix

The test set should be used for final evaluation only and should not be used during cross-validation or model selection.

Step 5 — Analyze and Improve

Follow the questions in studentID_netflix_report.docx to:

Explain how your model makes predictions.

Analyze False Positive and False Negative cases.

Make at least one reasonable modification to your modeling approach.

Compare the performance before and after your modification.

Complete the studentID_netflix_report.docx